# secretstorage smoke test

Tests for the `secretstorage` package — Python bindings to FreeDesktop.org Secret Service (D-Bus).

Most examples don't require a live D-Bus session bus; they exercise the importable API surface
and the exception hierarchy. Examples that need a bus will catch `SecretServiceNotAvailableException`
gracefully so the notebook still passes on systems without a running session bus.

In [ ]:
# Example 1: Public API surface — verify the primary callables/classes are exposed.
import secretstorage

expected_names = [
    'dbus_init',
    'Collection',
    'Item',
    'get_all_collections',
    'get_default_collection',
    'get_any_collection',
    'get_collection_by_alias',
    'create_collection',
    'search_items',
]

present = {n: hasattr(secretstorage, n) for n in expected_names}
for name, ok in present.items():
    print(f"  {name}: {'OK' if ok else 'MISSING'}")

missing = [n for n, ok in present.items() if not ok]
assert not missing, f"Missing expected names: {missing}"

In [ ]:
# Example 2: Exception hierarchy — secretstorage defines a base + several specific exceptions.
from secretstorage.exceptions import (
    SecretStorageException,
    SecretServiceNotAvailableException,
    LockedException,
    ItemNotFoundException,
    PromptDismissedException,
)

for exc in (
    SecretServiceNotAvailableException,
    LockedException,
    ItemNotFoundException,
    PromptDismissedException,
):
    assert issubclass(exc, SecretStorageException), f"{exc.__name__} should subclass SecretStorageException"
    print(f"  {exc.__name__} <- SecretStorageException OK")

# Instantiate one and confirm it formats.
e = ItemNotFoundException("no such item")
assert "no such item" in str(e)
print(f"\nException instantiation OK: {e!r}")

In [ ]:
# Example 3: dbus_init — attempt to connect to the session bus.
# This may fail on headless / CI systems with no D-Bus session bus; that's OK.
import secretstorage
from secretstorage.exceptions import SecretServiceNotAvailableException

bus = None
try:
    bus = secretstorage.dbus_init()
    print(f"Connected to D-Bus session bus: {type(bus).__name__}")
    print(f"  bus module: {type(bus).__module__}")
except SecretServiceNotAvailableException as e:
    print(f"No Secret Service available (expected in headless envs): {e}")
except Exception as e:
    # Other connection errors (no DBUS_SESSION_BUS_ADDRESS, etc.) — still acceptable in CI.
    print(f"D-Bus session not available ({type(e).__name__}): {str(e)[:120]}")
    raise

# Either outcome is fine for a smoke test; we only assert no unexpected crash type.
print("\ndbus_init call returned/raised cleanly")

In [ ]:
# Example 4: Collection / Item classes — verify they're proper classes with expected methods.
import inspect

from secretstorage import Collection, Item

# Collection — primary API methods
collection_methods = [
    'is_locked', 'unlock', 'lock', 'delete',
    'get_all_items', 'search_items', 'create_item',
    'get_label', 'set_label',
]
for m in collection_methods:
    assert hasattr(Collection, m), f"Collection missing method: {m}"
    assert callable(getattr(Collection, m)), f"Collection.{m} not callable"

# Item — primary API methods
item_methods = ['get_secret', 'set_secret', 'get_attributes', 'set_attributes', 'delete', 'get_label']
for m in item_methods:
    assert hasattr(Item, m), f"Item missing method: {m}"

assert inspect.isclass(Collection) and inspect.isclass(Item)
print(f"Collection methods OK: {collection_methods}")
print(f"Item methods OK: {item_methods}")

In [ ]:
# Example 5: search_items / get_all_collections signature + graceful failure without a bus.
import inspect
import os

import secretstorage
from secretstorage.exceptions import SecretServiceNotAvailableException

# Confirm function signatures take a bus connection as first arg.
sig_search = inspect.signature(secretstorage.search_items)
sig_all = inspect.signature(secretstorage.get_all_collections)
print(f"search_items{sig_search}")
print(f"get_all_collections{sig_all}")

assert 'connection' in sig_search.parameters or 'bus' in sig_search.parameters or len(sig_search.parameters) >= 2
assert len(sig_all.parameters) >= 1

_session_bus_address = os.environ.get("DBUS_SESSION_BUS_ADDRESS")
try:
    bus = secretstorage.dbus_init()
    collections = list(secretstorage.get_all_collections(bus))
    print(f"\nFound {len(collections)} collection(s) on this system")
    for c in collections[:3]:
        print(f"  - {type(c).__name__}")
except SecretServiceNotAvailableException as e:
    if _session_bus_address:
        raise
    _cause = e.__cause__
    assert isinstance(_cause, KeyError) and _cause.args == ("DBUS_SESSION_BUS_ADDRESS",), (
        f"no session bus is configured, yet secretstorage refused for another reason: {e!r} (cause {_cause!r})"
    )
    print(f"\nNo session bus here (expected without a session bus): {e}")

assert True  # smoke-test pass signal